# Object Detection Inference with DINOv3

This notebook demonstrates how to load the DINOv3 object detector, input a single image, and call the detection function to obtain bounding boxes, labels and scores.

## Setup

Let's start by loading some pre-requisites and checking the DINOv3 repository location:
- `local` if `DINOV3_LOCATION` environment variable was set to work with a local version of DINOv3 repository;
- `github` if the code should be loaded via torch hub.

In [ ]:
import os
import urllib

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from PIL import Image

import torch
from torchvision.transforms import v2

DINOV3_GITHUB_LOCATION = "facebookresearch/dinov3"

if os.getenv("DINOV3_LOCATION") is not None:
    DINOV3_LOCATION = os.getenv("DINOV3_LOCATION")
else:
    DINOV3_LOCATION = DINOV3_GITHUB_LOCATION

print(f"DINOv3 location set to {DINOV3_LOCATION}")

## Model Loading

We load the DINOv3 ViT-7B detection model pre-trained on COCO 2017.
The model takes a list of `(3, H, W)` normalized image tensors and returns a list of dicts,
each containing `"scores"`, `"labels"`, and `"boxes"` (XYXY format).

In [ ]:
model = torch.hub.load(
    repo_or_dir=DINOV3_LOCATION,
    model="dinov3_vit7b16_de",
    source="local" if DINOV3_LOCATION != DINOV3_GITHUB_LOCATION else "github",
)
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Model loaded on {device}")

## COCO Class Names

The model is trained on COCO 2017, which has 80 active object categories.
The class list below contains 91 entries (including background at index 0 and several reserved `N/A` slots)
to match the 1-indexed COCO label space used by the model output.
We define the class names here to decode the predicted labels.

In [ ]:
COCO_CLASSES = [
    "__background__",
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train",
    "truck", "boat", "traffic light", "fire hydrant", "N/A", "stop sign",
    "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep",
    "cow", "elephant", "bear", "zebra", "giraffe", "N/A", "backpack",
    "umbrella", "N/A", "N/A", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat",
    "baseball glove", "skateboard", "surfboard", "tennis racket", "bottle",
    "N/A", "wine glass", "cup", "fork", "knife", "spoon", "bowl",
    "banana", "apple", "sandwich", "orange", "broccoli", "carrot",
    "hot dog", "pizza", "donut", "cake", "chair", "couch", "potted plant",
    "bed", "N/A", "dining table", "N/A", "N/A", "toilet", "N/A", "tv",
    "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave",
    "oven", "toaster", "sink", "refrigerator", "N/A", "book", "clock",
    "vase", "scissors", "teddy bear", "hair drier", "toothbrush",
]

## Image Loading and Preprocessing

We load a sample image and apply the standard ImageNet normalization transform.
The model expects `(3, H, W)` float tensors normalized with ImageNet mean and std.

In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def make_detection_transform():
    return v2.Compose([
        v2.ToImage(),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def load_image_from_url(url: str) -> Image.Image:
    with urllib.request.urlopen(url) as f:
        return Image.open(f).convert("RGB")


EXAMPLE_IMAGE_URL = "https://dl.fbaipublicfiles.com/dinov2/images/example.jpg"
image_pil = load_image_from_url(EXAMPLE_IMAGE_URL)

transform = make_detection_transform()
image_tensor = transform(image_pil)  # (3, H, W)
print(f"Image tensor shape: {image_tensor.shape}")

plt.figure(figsize=(8, 6))
plt.imshow(image_pil)
plt.axis("off")
plt.title("Input image")
plt.show()

## Running Detection

We pass the image tensor to the detection model.
The model accepts a **list** of `(3, H, W)` normalized tensors and returns a list of dicts
with `"scores"`, `"labels"` (COCO class indices), and `"boxes"` (XYXY pixel coordinates).

In [ ]:
SCORE_THRESHOLD = 0.5  # Only show detections with confidence above this threshold

with torch.inference_mode():
    detections = model([image_tensor.to(device)])

# detections is a list with one entry per input image
result = detections[0]
scores = result["scores"].cpu()
labels = result["labels"].cpu()
boxes = result["boxes"].cpu()

# Filter by score threshold
keep = scores >= SCORE_THRESHOLD
scores = scores[keep]
labels = labels[keep]
boxes = boxes[keep]

print(f"Detections above threshold {SCORE_THRESHOLD}: {keep.sum().item()}")
for score, label, box in zip(scores, labels, boxes):
    class_name = COCO_CLASSES[label.item()] if label.item() < len(COCO_CLASSES) else "unknown"
    x1, y1, x2, y2 = box.tolist()
    print(f"  {class_name} ({score:.2f}): [{x1:.0f}, {y1:.0f}, {x2:.0f}, {y2:.0f}]")

## Visualizing Results

We draw the predicted bounding boxes and class labels on the original image.

In [ ]:
fig, ax = plt.subplots(1, figsize=(12, 8))
ax.imshow(image_pil)

cmap = plt.get_cmap("tab20")
unique_labels = labels.unique().tolist()
color_map = {lbl: cmap(i / max(len(unique_labels), 1)) for i, lbl in enumerate(unique_labels)}

for score, label, box in zip(scores, labels, boxes):
    x1, y1, x2, y2 = box.tolist()
    class_name = COCO_CLASSES[label.item()] if label.item() < len(COCO_CLASSES) else "unknown"
    color = color_map[label.item()]

    rect = patches.Rectangle(
        (x1, y1), x2 - x1, y2 - y1,
        linewidth=2, edgecolor=color, facecolor="none",
    )
    ax.add_patch(rect)
    ax.text(
        x1, y1 - 4,
        f"{class_name} {score:.2f}",
        color="white", fontsize=8,
        bbox=dict(facecolor=color, alpha=0.8, pad=1, edgecolor="none"),
    )

ax.axis("off")
ax.set_title(f"Detections (threshold={SCORE_THRESHOLD})")
plt.tight_layout()
plt.show()